# 04 — Klasifikimi dhe regresioni logjistik

## Konteksti dhe problemi shkencor

Pulse detektori përdoren për të lidhur probabilitetin, kufirin linear, metrikat dhe zgjedhjen e pragut me një problem shkencor.

### Objektivat e të nxënit

- të implementoni regresionin logjistik;
- të përdorni confusion matrix, ROC dhe calibration;
- të kuptoni pragun e vendimit;
- të interpretoni koeficientët me kujdes.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Diskriminimi sinjal–sfond

In [ ]:
df=pd.read_csv(DATA/'detector_pulses.csv'); X=df[['amplitude','width_us']].to_numpy(); y=df.class_signal.to_numpy(); plt.scatter(X[:,0],X[:,1],c=y,s=15); plt.show()

## 2. Sigmoidi dhe cross-entropy

In [ ]:
z=np.linspace(-8,8,300); sig=1/(1+np.exp(-z)); plt.plot(z,sig); plt.axhline(.5,ls='--'); plt.show()

Për $p_i=\sigma(w^Tx_i+b)$ minimizojmë $$-\frac1N\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)].$$

## 3. Implementim manual me gradient descent

In [ ]:
from src.classification.manual import logistic_fit_gd,logistic_predict_proba
sc=StandardScaler(); Xs=sc.fit_transform(X); w,h=logistic_fit_gd(Xs,y,lr=.15,n_iter=1500,l2=.001); print(w); plt.plot(h); plt.yscale('log'); plt.show()

In [ ]:
p=logistic_predict_proba(Xs,w); pred=(p>=.5).astype(int); print('accuracy manual=',np.mean(pred==y))

## 4. Krahasim me scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression
sk=LogisticRegression(C=1000).fit(Xs,y); print(sk.intercept_,sk.coef_); print(sk.score(Xs,y))

## 5. Kufiri i vendimit

In [ ]:
xx,yy=np.meshgrid(np.linspace(X[:,0].min()-.1,X[:,0].max()+.1,250),np.linspace(X[:,1].min()-.05,X[:,1].max()+.05,250)); grid=np.c_[xx.ravel(),yy.ravel()]; zz=sk.predict_proba(sc.transform(grid))[:,1].reshape(xx.shape); plt.scatter(X[:,0],X[:,1],c=y,s=12,alpha=.5); plt.contour(xx,yy,zz,levels=[.5]); plt.show()

## 6. Matrica e konfuzionit dhe pragjet

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report,roc_curve,roc_auc_score,precision_recall_curve
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=0); pipe=make_pipeline(StandardScaler(),LogisticRegression()).fit(Xtr,ytr); prob=pipe.predict_proba(Xte)[:,1]
for th in [.3,.5,.7]: print(th,confusion_matrix(yte,(prob>=th).astype(int)).tolist())

## 7. ROC dhe Precision–Recall

In [ ]:
fpr,tpr,_=roc_curve(yte,prob); pr,rc,_=precision_recall_curve(yte,prob); fig,ax=plt.subplots(); ax.plot(fpr,tpr,label=f'ROC AUC={roc_auc_score(yte,prob):.3f}'); ax.plot([0,1],[0,1],'--'); ax.legend(); plt.show(); plt.figure(); plt.plot(rc,pr); plt.xlabel('recall'); plt.ylabel('precision'); plt.show()

## 8. Rast i rrallë: pse accuracy dështon

In [ ]:
yr=np.r_[np.zeros(990,dtype=int),np.ones(10,dtype=int)]; pred0=np.zeros_like(yr); print('accuracy e modelit absurd=',np.mean(pred0==yr))

## 9. Kalibrimi

In [ ]:
from sklearn.calibration import calibration_curve
frac,mean=calibration_curve(yte,prob,n_bins=8); plt.plot(mean,frac,marker='o'); plt.plot([0,1],[0,1],'--'); plt.xlabel('p e parashikuar'); plt.ylabel('frekuenca reale'); plt.show()

## 10. Ushtrime

- Derivoni gradientin.
- Zgjidhni pragun kur FN kushton 5 herë më shumë se FP.
- Krahasoni ROC-AUC dhe PR-AUC kur klasa pozitive bëhet 5%.
- Shtoni regularizim dhe interpretoni ndryshimin e koeficientëve.

## 11. Shembull i plotë

Shkruani një raport sinjal/sfond ku zgjidhni pragun sipas një objektivi shkencor të deklaruar, jo sipas accuracy.

## 12. Playground

Ndryshoni mbivendosjen e klasave duke shtuar zhurmë dhe vëzhgoni si deformohen ROC, precision-recall dhe kufiri i vendimit.